In [ ]:
# Block 2: Imports
import os
import glob
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import cv2
import rasterio
from rasterio.windows import Window

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, jaccard_score, precision_score, recall_score

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, UpSampling2D,
                                     Concatenate, BatchNormalization, Activation,
                                     Multiply, Add)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Block 3: CPU-friendly config
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PATCH_SIZE = 128          # CHANGED: was 64 — too small, loses forest context
STRIDE     = 64           # CHANGED: was 128 — 50% overlap for better coverage
BATCH_SIZE = 8            # CHANGED: was 4 — safe to increase with 128x128
EPOCHS     = 30           # CHANGED: was 15 — more room with lower LR + scheduler
N_CLUSTERS = 2
LEARNING_RATE = 1e-4      # NEW: centralised here instead of hardcoded in compile

DATA_DIR   = './data'
OUTPUT_DIR = './hasdeo_outputs'
MODEL_DIR  = os.path.join(OUTPUT_DIR, 'models')
PRED_DIR   = os.path.join(OUTPUT_DIR, 'predictions')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(PRED_DIR,   exist_ok=True)

In [ ]:
# Block 4: Discover yearly TIFF files
tif_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.tif'))) + sorted(glob.glob(os.path.join(DATA_DIR, '*.tiff')))
print('Found files:')
for f in tif_files:
    print('-', os.path.basename(f))

assert len(tif_files) > 0, 'No .tif files found. Put year-wise TIFFs inside ./hasdeo_tifs/'

In [ ]:
# Block 6: Utility functions
import math
def minmax_normalize(arr):
    arr = arr.astype(np.float32)
    out = np.zeros_like(arr, dtype=np.float32)
    for c in range(arr.shape[-1]):
        band = arr[..., c]
        mn, mx = np.nanmin(band), np.nanmax(band)
        if mx > mn:
            out[..., c] = (band - mn) / (mx - mn)
        else:
            out[..., c] = 0
    return np.nan_to_num(out)

def read_tif_as_hwc(path, max_bands=4):
    # kept for small reads / sanity checks only
    with rasterio.open(path) as src:
        count = min(src.count, max_bands)
        arr   = src.read(list(range(1, count + 1)))
    return np.transpose(arr, (1, 2, 0))

# NEW: read one patch-sized window at a time — never loads full image
def extract_patches_windowed(path, patch_size=128, stride=64, max_bands=4):
    patches = []
    coords  = []
    with rasterio.open(path) as src:
        h, w   = src.height, src.width
        count  = min(src.count, max_bands)
        bands  = list(range(1, count + 1))

        for y in range(0, h - patch_size + 1, stride):
            for x in range(0, w - patch_size + 1, stride):
                window = Window(x, y, patch_size, patch_size)
                arr    = src.read(bands, window=window)          # shape: (C, H, W)
                arr    = np.transpose(arr, (1, 2, 0))            # → (H, W, C)
                arr    = arr.astype(np.float32)

                # per-patch normalization
                for c in range(arr.shape[-1]):
                    band   = arr[..., c]
                    mn, mx = np.nanmin(band), np.nanmax(band)
                    if mx > mn:
                        arr[..., c] = (band - mn) / (mx - mn)
                    else:
                        arr[..., c] = 0
                arr = np.nan_to_num(arr)

                patches.append(arr)
                coords.append((x, y))

    return patches, coords   # list of arrays, not one big array

def get_pixel_area_ha(path):
    with rasterio.open(path) as src:
        res_x, res_y = src.res
    lat = 23.1
    m_per_deg_lat = 111320
    m_per_deg_lon = 111320 * math.cos(math.radians(lat))

    pixel_area_m2 = abs(res_x * m_per_deg_lon) * abs(res_y * m_per_deg_lat)
    pixel_area_ha = pixel_area_m2 / 10000
    return pixel_area_ha

def show_patch(img, title='image', cmap=None):
    plt.figure(figsize=(4, 4))
    if img.ndim == 2:
        plt.imshow(img, cmap=cmap)
    else:
        plt.imshow(img[..., :3])
    plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
def generate_mask_kmeans_refined(
    patch,
    n_clusters=2,
    smooth=True,
    blur_kernel=5,
    morph_kernel=3,
    iterations=1,
    use_nir_rule=True
):
    """
    Multi-index pseudo-ground-truth mask generator.
    Uses NDVI + EVI + SAVI + NDWI + morphology.

    Parameters
    ----------
    patch : np.ndarray
        Image patch of shape (H, W, C).
        Expected 4-band order: [R, G, B, NIR]

    n_clusters : int
        Kept only for compatibility; not used.
    smooth : bool
        Whether to smooth the fused vegetation score before thresholding.
    blur_kernel : int
        Kernel size for Gaussian blur, should be odd.
    morph_kernel : int
        Kernel size for morphological cleanup.
    iterations : int
        Morphology iterations.
    use_nir_rule : bool
        Kept only for compatibility; not used.

    Returns
    -------
    mask : np.ndarray
        Binary mask of shape (H, W, 1), dtype float32.
        1 = forest, 0 = non-forest
    """
    patch = patch.astype(np.float32)

    if patch.ndim != 3 or patch.shape[-1] < 4:
        raise ValueError("This fused-index version requires at least 4 channels in [R, G, B, NIR] order.")

    # Normalize each band to [0, 1]
    pmin = patch.min(axis=(0, 1), keepdims=True)
    pmax = patch.max(axis=(0, 1), keepdims=True)
    patch_norm = (patch - pmin) / (pmax - pmin + 1e-8)

    # Assumed band order: [R, G, B, NIR]
    red   = patch_norm[..., 0]
    green = patch_norm[..., 1]
    blue  = patch_norm[..., 2]
    nir   = patch_norm[..., 3]

    # Spectral indices
    ndvi = (nir - red) / (nir + red + 1e-8)

    evi = 2.5 * ((nir - red) / (nir + 6.0 * red - 7.5 * blue + 1.0 + 1e-8))

    L = 0.5
    savi = ((nir - red) / (nir + red + L + 1e-8)) * (1.0 + L)

    # McFeeters NDWI: (Green - NIR) / (Green + NIR)
    # Higher values tend to indicate water/moisture, so we subtract it in the fusion score
    ndwi = (green - nir) / (green + nir + 1e-8)

    # Clip to avoid extreme values from dominating
    ndvi = np.clip(ndvi, -1.0, 1.0)
    evi  = np.clip(evi,  -1.0, 1.0)
    savi = np.clip(savi, -1.0, 1.0)
    ndwi = np.clip(ndwi, -1.0, 1.0)

    # Fused vegetation score
    veg_score = (
        0.40 * ndvi +
        0.30 * evi +
        0.20 * savi -
        0.10 * ndwi
    )

    # Optional smoothing
    if smooth:
        veg_score = cv2.GaussianBlur(veg_score, (blur_kernel, blur_kernel), 0)

    # Threshold fused score
    # Start here, then tune slightly if needed
    mask = (veg_score > 0.25).astype(np.uint8)

    # Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_kernel, morph_kernel))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=iterations)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=iterations)

    return mask[..., None].astype(np.float32)

In [ ]:
# Block 8: Extract patches via sliding window and save to disk
import gc

PATCH_DIR = os.path.join(OUTPUT_DIR, 'patches')
os.makedirs(PATCH_DIR, exist_ok=True)

patch_index = []

for path in tif_files:
    year     = os.path.splitext(os.path.basename(path))[0]
    year_dir = os.path.join(PATCH_DIR, year)
    os.makedirs(year_dir, exist_ok=True)

    print(f'Processing {year}...', end=' ')

    # CHANGED: windowed read — never loads full image
    patches, coords = extract_patches_windowed(
        path,
        patch_size=PATCH_SIZE,
        stride=STRIDE,
        max_bands=4
    )

    for i, patch in enumerate(patches):
        mask = generate_mask_kmeans_refined(patch, n_clusters=N_CLUSTERS)

        img_path  = os.path.join(year_dir, f'img_{i:05d}.npy')
        mask_path = os.path.join(year_dir, f'mask_{i:05d}.npy')

        np.save(img_path,  patch.astype(np.float32))
        np.save(mask_path, mask.astype(np.float32))

        patch_index.append((img_path, mask_path, year))

    # discard patch list immediately after saving
    del patches, coords
    gc.collect()
    print(f'done — {len([p for p in patch_index if p[2]==year])} patches saved')

index_df = pd.DataFrame(patch_index, columns=['img_path', 'mask_path', 'year'])
index_df.to_csv(os.path.join(OUTPUT_DIR, 'patch_index.csv'), index=False)

print(f'\nTotal patches saved: {len(patch_index)}')
print(index_df['year'].value_counts().sort_index())

In [ ]:
# Block 9: Visual sanity check
sample = patch_index[np.random.randint(0, len(patch_index))]
img_s  = np.load(sample[0])
mask_s = np.load(sample[1])

show_patch(img_s,          f'Patch — year {sample[2]}')
show_patch(mask_s.squeeze(), 'Generated mask', cmap='gray')

In [ ]:
# Block 10: Train-validation split by year (index only — no RAM spike)
import pandas as pd
import os

index_df    = pd.read_csv(os.path.join(OUTPUT_DIR, 'patch_index.csv'))
unique_years = sorted(index_df['year'].unique().tolist())
print('All years found:', unique_years)

val_years   = unique_years[-2:]
train_years = unique_years[:-2]
print('Training years:  ', train_years)
print('Validation years:', val_years)

train_df = index_df[index_df['year'].isin(train_years)].reset_index(drop=True)
val_df   = index_df[index_df['year'].isin(val_years)].reset_index(drop=True)

print(f'Train patches: {len(train_df)}')
print(f'Val patches:   {len(val_df)}')

In [ ]:
# Block 10B: Disk-based data generator (no full array in RAM)
import numpy as np

def augment_numpy(image, mask):
    if np.random.rand() > 0.5:
        image = np.fliplr(image);  mask = np.fliplr(mask)
    if np.random.rand() > 0.5:
        image = np.flipud(image);  mask = np.flipud(mask)
    k     = np.random.randint(0, 4)
    image = np.rot90(image, k);    mask = np.rot90(mask, k)
    delta = np.random.uniform(-0.1, 0.1)
    image = np.clip(image + delta, 0.0, 1.0).astype(np.float32)
    return image, mask

def disk_generator(df, batch_size, augment=False, shuffle=True):
    indices = np.arange(len(df))
    while True:
        if shuffle:
            np.random.shuffle(indices)
        for start in range(0, len(indices) - batch_size + 1, batch_size):
            batch_idx = indices[start:start + batch_size]
            X_batch, y_batch = [], []
            for i in batch_idx:
                row = df.iloc[i]
                img  = np.load(row['img_path'])
                mask = np.load(row['mask_path'])
                if augment:
                    img, mask = augment_numpy(img, mask)
                X_batch.append(img)
                y_batch.append(mask)
            yield (np.array(X_batch, dtype=np.float32),
                   np.array(y_batch, dtype=np.float32))

STEPS_PER_EPOCH  = len(train_df) // BATCH_SIZE
VALIDATION_STEPS = len(val_df)   // BATCH_SIZE

train_gen = disk_generator(train_df, BATCH_SIZE, augment=True,  shuffle=True)
val_gen   = disk_generator(val_df,   BATCH_SIZE, augment=False, shuffle=False)

print(f'Steps per epoch:  {STEPS_PER_EPOCH}')
print(f'Validation steps: {VALIDATION_STEPS}')

In [ ]:
# Block 11: Attention gate and model blocks
from tensorflow.keras.layers import Conv2DTranspose, Dropout
from tensorflow.keras.regularizers import l2

REG = l2(1e-4)   # NEW: L2 regularization applied to all conv layers

def conv_block(x, filters, dropout_rate=0.3):
    # CHANGED: added kernel_regularizer and Dropout
    x = Conv2D(filters, 3, padding='same', kernel_regularizer=REG)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(dropout_rate)(x)          # NEW
    x = Conv2D(filters, 3, padding='same', kernel_regularizer=REG)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x

def attention_gate(x, g, inter_channels):
    theta_x = Conv2D(inter_channels, 1, padding='same')(x)
    phi_g   = Conv2D(inter_channels, 1, padding='same')(g)
    add     = Add()([theta_x, phi_g])
    act     = Activation('relu')(add)
    psi     = Conv2D(1, 1, padding='same')(act)
    psi     = Activation('sigmoid')(psi)
    out     = Multiply()([x, psi])
    return out

def build_attention_unet(input_shape):
    inputs = Input(shape=input_shape)

    c1 = conv_block(inputs, 16)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = conv_block(p1, 32)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = conv_block(p2, 64)
    p3 = MaxPooling2D((2, 2))(c3)

    c4 = conv_block(p3, 128)
    p4 = MaxPooling2D((2, 2))(c4)

    bn = conv_block(p4, 256, dropout_rate=0.4)   # slightly higher dropout at bottleneck

    # CHANGED: UpSampling2D → Conv2DTranspose (learnable upsampling)
    u4 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(bn)
    a4 = attention_gate(c4, u4, 128)
    u4 = Concatenate()([u4, a4])
    c5 = conv_block(u4, 128)

    u3 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c5)
    a3 = attention_gate(c3, u3, 64)
    u3 = Concatenate()([u3, a3])
    c6 = conv_block(u3, 64)

    u2 = Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c6)
    a2 = attention_gate(c2, u2, 32)
    u2 = Concatenate()([u2, a2])
    c7 = conv_block(u2, 32)

    u1 = Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same')(c7)
    a1 = attention_gate(c1, u1, 16)
    u1 = Concatenate()([u1, a1])
    c8 = conv_block(u1, 16)

    outputs = Conv2D(1, 1, activation='sigmoid')(c8)
    return Model(inputs, outputs, name='AttentionUNet_Improved')

In [ ]:
# Block 12: Metrics and compile

# CHANGED: X no longer exists — read n_bands from a saved patch file
sample_patch = np.load(patch_index[0][0])
N_BANDS      = sample_patch.shape[-1]
print(f'Bands detected: {N_BANDS}')
del sample_patch

def iou_metric(y_true, y_pred, smooth=1e-6):
    y_pred       = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    union        = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection
    return (intersection + smooth) / (union + smooth)

def dice_loss(y_true, y_pred, smooth=1e-6):
    intersection = tf.reduce_sum(y_true * y_pred)
    return 1 - (2. * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

def bce_dice_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

model = build_attention_unet((PATCH_SIZE, PATCH_SIZE, N_BANDS))

model.compile(
    optimizer=Adam(LEARNING_RATE),
    loss=bce_dice_loss,
    metrics=['accuracy', iou_metric]
)
model.summary()

In [ ]:
# Block 13: Callbacks
from tensorflow.keras.callbacks import ReduceLROnPlateau   # NEW

checkpoint_path = os.path.join(MODEL_DIR, 'best_attention_unet.keras')

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=7,                  # CHANGED: was 5 — give more room with lower LR
        restore_best_weights=True,
        min_delta=0.001
    ),
    ModelCheckpoint(
        checkpoint_path,
        monitor='val_loss',
        save_best_only=True
    ),
    # NEW: halve LR when val_loss plateaus for 3 epochs
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    CSVLogger(os.path.join(OUTPUT_DIR, 'training_log.csv'))
]

In [ ]:
# Block 14: Train model
history = model.fit(
    train_gen,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_gen,
    validation_steps=VALIDATION_STEPS,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Block 15: Plot training curves
hist_df = pd.DataFrame(history.history)
hist_df.to_csv(os.path.join(OUTPUT_DIR, 'history.csv'), index=False)

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(hist_df['loss'], label='train_loss')
plt.plot(hist_df['val_loss'], label='val_loss')
plt.legend()
plt.title('Loss')

plt.subplot(1,2,2)
plt.plot(hist_df['iou_metric'], label='train_iou')
plt.plot(hist_df['val_iou_metric'], label='val_iou')
plt.legend()
plt.title('IoU')
plt.tight_layout()
plt.show()

In [ ]:
# Block 16: Evaluate on validation set (fixed — use disk generator)
import math

# Compute number of validation steps (covers remainder)
val_steps = int(np.ceil(len(val_df) / BATCH_SIZE)) if len(val_df) > 0 else 1

# Create a fresh generator for evaluation (no shuffling, no augmentation)
val_eval_gen = disk_generator(val_df, BATCH_SIZE, augment=False, shuffle=False)

y_batches = []
pred_batches = []

for _ in range(val_steps):
    Xb, yb = next(val_eval_gen)
    preds_b = model.predict(Xb, batch_size=BATCH_SIZE, verbose=0)
    y_batches.append(yb)
    pred_batches.append(preds_b)

preds = np.vstack(pred_batches)[: sum([b.shape[0] for b in y_batches])]
pred_bin = (preds > 0.5).astype(np.uint8)

y_val = np.vstack(y_batches)[:preds.shape[0]]

yt = y_val.reshape(-1).astype(np.uint8)
yp = pred_bin.reshape(-1).astype(np.uint8)

metrics = {
    'f1': f1_score(yt, yp, zero_division=0),
    'precision': precision_score(yt, yp, zero_division=0),
    'recall': recall_score(yt, yp, zero_division=0),
    'iou': jaccard_score(yt, yp, zero_division=0)
}

metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'validation_metrics.csv'), index=False)
metrics_df

In [ ]:
# Block 16B: ROC-AUC score + ROC curve + Confusion Matrix
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay

yt_flat = y_val.reshape(-1).astype(np.uint8)
yp_prob  = preds.reshape(-1)          # raw probabilities, not binarized
yp_bin   = pred_bin.reshape(-1).astype(np.uint8)

# ── ROC-AUC
roc_auc = roc_auc_score(yt_flat, yp_prob)
fpr, tpr, thresholds = roc_curve(yt_flat, yp_prob)

print(f"ROC-AUC Score: {roc_auc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random classifier')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='darkorange')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curve — Forest vs Non-Forest', fontsize=13, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# Confusion Matrix
cm = confusion_matrix(yt_flat, yp_bin)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Forest', 'Forest'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix — Validation Set', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'roc_confusion.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved roc_confusion.png")


In [ ]:
# Block 16C: Full model performance summary table
from IPython.display import display

# Pixel-level metrics already computed above
tn, fp, fn, tp = confusion_matrix(yt_flat, yp_bin).ravel()
specificity = tn / (tn + fp + 1e-9)
balanced_acc = (tp / (tp + fn + 1e-9) + specificity) / 2

summary = {
    'Metric': ['Precision', 'Recall (Sensitivity)', 'Specificity',
               'F1 Score', 'IoU (Jaccard)', 'ROC-AUC', 'Balanced Accuracy',
               'True Positives', 'True Negatives', 'False Positives', 'False Negatives'],
    'Value':  [
        f"{precision_score(yt_flat, yp_bin, zero_division=0):.4f}",
        f"{recall_score(yt_flat, yp_bin, zero_division=0):.4f}",
        f"{specificity:.4f}",
        f"{f1_score(yt_flat, yp_bin, zero_division=0):.4f}",
        f"{jaccard_score(yt_flat, yp_bin, zero_division=0):.4f}",
        f"{roc_auc:.4f}",
        f"{balanced_acc:.4f}",
        str(int(tp)), str(int(tn)), str(int(fp)), str(int(fn))
    ]
}

summary_df = pd.DataFrame(summary)
summary_df.to_csv(os.path.join(OUTPUT_DIR, 'full_metrics_summary.csv'), index=False)
print("\n========== MODEL PERFORMANCE SUMMARY ==========")
display(summary_df.style
    .set_properties(**{'text-align': 'left'})
    .set_table_styles([{'selector': 'th', 'props': [('font-weight', 'bold'), ('font-size', '13px')]}])
)
print("Saved full_metrics_summary.csv")


In [ ]:
# Block 17: Visualize predictions (fixed — use y_val and pred_bin from evaluation)
for i in range(3):
    idx = np.random.randint(0, len(y_val))
    
    # Reconstruct the input patch from the validation dataframe
    sample_row = val_df.iloc[idx % len(val_df)]
    X_patch = np.load(sample_row['img_path'])
    
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1)
    plt.imshow(X_patch[..., :3])
    plt.title('Input patch')
    plt.axis('off')

    plt.subplot(1,3,2)
    plt.imshow(y_val[idx].squeeze(), cmap='gray')
    plt.title('Ground truth mask')
    plt.axis('off')

    plt.subplot(1,3,3)
    plt.imshow(pred_bin[idx].squeeze(), cmap='gray')
    plt.title('Model prediction')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Block 18: Predict an entire year image patch-by-patch
def reconstruct_from_patches(pred_patches, coords, image_shape, patch_size=128):
    h, w = image_shape[:2]
    out = np.zeros((h, w), dtype=np.float32)
    count = np.zeros((h, w), dtype=np.float32)

    for pred, (x, y) in zip(pred_patches, coords):
        pred2d = pred.squeeze()
        out[y:y+patch_size, x:x+patch_size] += pred2d
        count[y:y+patch_size, x:x+patch_size] += 1

    count[count == 0] = 1
    out = out / count
    return out

def predict_year_map(tif_path):
    img = read_tif_as_hwc(tif_path, max_bands=4)
    img = minmax_normalize(img)
    patches, coords = extract_patches_windowed(tif_path, patch_size=PATCH_SIZE, stride=STRIDE, max_bands=4)
    pred = model.predict(np.array(patches), batch_size=BATCH_SIZE, verbose=0)
    recon = reconstruct_from_patches(pred, coords, img.shape, patch_size=PATCH_SIZE)
    binary = (recon > 0.5).astype(np.uint8)
    return img, recon, binary

In [ ]:
# Block 19: Run prediction for 2018 and 2024
path_2018 = os.path.join(DATA_DIR, '2018.tif')
path_2024 = os.path.join(DATA_DIR, '2024.tif')

if os.path.exists(path_2018) and os.path.exists(path_2024):
    img_2018, prob_2018, bin_2018 = predict_year_map(path_2018)
    img_2024, prob_2024, bin_2024 = predict_year_map(path_2024)

    np.save(os.path.join(PRED_DIR, 'prob_2018.npy'), prob_2018)
    np.save(os.path.join(PRED_DIR, 'prob_2024.npy'), prob_2024)
    np.save(os.path.join(PRED_DIR, 'bin_2018.npy'), bin_2018)
    np.save(os.path.join(PRED_DIR, 'bin_2024.npy'), bin_2024)

    print('Saved 2018 and 2024 predictions.')
else:
    print('2018.tif or 2024.tif not found; rename your files accordingly.')

In [ ]:
# Block 20: Simple change map (forest loss proxy)
if 'bin_2018' in globals() and 'bin_2024' in globals():
    forest_loss = ((bin_2018 == 1) & (bin_2024 == 0)).astype(np.uint8)
    forest_gain = ((bin_2018 == 0) & (bin_2024 == 1)).astype(np.uint8)

    np.save(os.path.join(PRED_DIR, 'forest_loss_2018_2024.npy'), forest_loss)
    np.save(os.path.join(PRED_DIR, 'forest_gain_2018_2024.npy'), forest_gain)

    plt.figure(figsize=(15,5))
    plt.subplot(1,3,1)
    plt.imshow(bin_2018, cmap='gray')
    plt.title('Predicted forest mask 2018')
    plt.axis('off')

    plt.subplot(1,3,2)
    plt.imshow(bin_2024, cmap='gray')
    plt.title('Predicted forest mask 2024')
    plt.axis('off')

    plt.subplot(1,3,3)
    plt.imshow(forest_loss, cmap='hot')
    plt.title('Forest loss 2018 to 2024')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Block 21: Export predicted raster aligned to a source TIFF
def save_mask_like_reference(mask2d, reference_tif, out_tif):
    with rasterio.open(reference_tif) as src:
        profile = src.profile.copy()
        profile.update(count=1, dtype=rasterio.uint8)
        with rasterio.open(out_tif, 'w', **profile) as dst:
            dst.write(mask2d.astype(rasterio.uint8), 1)

if 'bin_2024' in globals() and os.path.exists(path_2024):
    save_mask_like_reference(bin_2024, path_2024, os.path.join(PRED_DIR, 'predicted_mask_2024.tif'))
    print('Saved predicted_mask_2024.tif')

In [ ]:
# Block 22: Per-year forest coverage statistics
from IPython.display import display

year_stats = []

print(f'Validation years: {val_years}')
print(f'Processing all years for statistics...\n')

for path in tqdm(tif_files, desc='Computing per-year forest coverage'):
    year = os.path.splitext(os.path.basename(path))[0]

    # Compute for validation years only (predictions on training data are not trustworthy)
    if year not in val_years:
        print(f'Skipping {year} (training data)')
        continue

    img, prob_map, bin_map = predict_year_map(path)
    total_pixels    = bin_map.size
    forest_pixels   = int(bin_map.sum())
    nonforest_pixels = total_pixels - forest_pixels
    forest_pct      = 100.0 * forest_pixels / total_pixels

    # Read actual pixel resolution from file metadata
    pixel_area_ha  = get_pixel_area_ha(path)
    forest_area_ha = forest_pixels * pixel_area_ha

    year_stats.append({
        'Year':                  year,
        'Total Pixels':          total_pixels,
        'Forest Pixels':         forest_pixels,
        'Non-Forest Pixels':     nonforest_pixels,
        'Forest Coverage (%)':   round(forest_pct, 2),
        'Est. Forest Area (ha)': round(forest_area_ha, 1)
    })

if len(year_stats) == 0:
    print('⚠️  WARNING: No validation years processed. Using all years for statistics instead.')
    # Fallback: compute for all years
    for path in tqdm(tif_files, desc='Computing per-year forest coverage (all years)'):
        year = os.path.splitext(os.path.basename(path))[0]
        img, prob_map, bin_map = predict_year_map(path)
        total_pixels    = bin_map.size
        forest_pixels   = int(bin_map.sum())
        nonforest_pixels = total_pixels - forest_pixels
        forest_pct      = 100.0 * forest_pixels / total_pixels
        pixel_area_ha  = get_pixel_area_ha(path)
        forest_area_ha = forest_pixels * pixel_area_ha
        year_stats.append({
            'Year':                  year,
            'Total Pixels':          total_pixels,
            'Forest Pixels':         forest_pixels,
            'Non-Forest Pixels':     nonforest_pixels,
            'Forest Coverage (%)':   round(forest_pct, 2),
            'Est. Forest Area (ha)': round(forest_area_ha, 1)
        })

year_df = pd.DataFrame(year_stats).sort_values('Year').reset_index(drop=True)
year_df.to_csv(os.path.join(OUTPUT_DIR, 'per_year_forest_coverage.csv'), index=False)

print('\n========== PER-YEAR FOREST COVERAGE ==========')
display(year_df)

In [ ]:
# Block 23: Year-over-year deforestation rate table + chart
from IPython.display import display

defo_rows = []
years_sorted = year_df['Year'].tolist()

for i in range(1, len(year_df)):
    prev = year_df.iloc[i - 1]
    curr = year_df.iloc[i]

    forest_lost_px = max(0, prev['Forest Pixels'] - curr['Forest Pixels'])
    forest_lost_ha = forest_lost_px * (10 * 10 / 10000)
    defo_rate_pct  = 100.0 * forest_lost_px / (prev['Forest Pixels'] + 1e-9)
    net_change_ha  = curr['Est. Forest Area (ha)'] - prev['Est. Forest Area (ha)']

    defo_rows.append({
        'Period':                  f"{prev['Year']} → {curr['Year']}",
        'Forest Lost (pixels)':    int(forest_lost_px),
        'Forest Lost (ha)':        round(forest_lost_ha, 1),
        'Net Forest Change (ha)':  round(net_change_ha, 1),
        'Deforestation Rate (%)':  round(defo_rate_pct, 2),
    })

defo_df = pd.DataFrame(defo_rows)
defo_df.to_csv(os.path.join(OUTPUT_DIR, 'deforestation_rates.csv'), index=False)

print("\n========== DEFORESTATION RATE TABLE ==========")
display(defo_df)

# ── Bar chart: forest coverage % over years
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(year_df['Year'], year_df['Forest Coverage (%)'],
            color=['#2d6a4f' if v >= year_df['Forest Coverage (%)'].max() * 0.9 else '#95d5b2'
                   for v in year_df['Forest Coverage (%)']],
            edgecolor='black', linewidth=0.6)
axes[0].set_title('Forest Coverage (%) by Year', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Forest Coverage (%)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(axes[0].patches, year_df['Forest Coverage (%)']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

# ── Bar chart: deforestation rate % per period
colors = ['#d62728' if v > 0 else '#2ca02c' for v in defo_df['Deforestation Rate (%)']]
axes[1].bar(defo_df['Period'], defo_df['Deforestation Rate (%)'],
            color=colors, edgecolor='black', linewidth=0.6)
axes[1].set_title('Annual Deforestation Rate (%) by Period', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Period')
axes[1].set_ylabel('Deforestation Rate (%)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
for bar, val in zip(axes[1].patches, defo_df['Deforestation Rate (%)']):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + (0.1 if val >= 0 else -0.4),
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'deforestation_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved deforestation_chart.png")


In [ ]:
# Block 24: Cumulative forest loss trend + summary stats
from IPython.display import display
import numpy as np

baseline_ha = year_df.iloc[0]['Est. Forest Area (ha)']
year_df_plot = year_df.copy()
year_df_plot['Cumulative Loss (ha)']    = baseline_ha - year_df_plot['Est. Forest Area (ha)']
year_df_plot['Cumulative Loss (%)']     = 100.0 * year_df_plot['Cumulative Loss (ha)'] / (baseline_ha + 1e-9)

# ── Linear trend fit
x = np.arange(len(year_df_plot))
y_vals = year_df_plot['Forest Coverage (%)'].values
coeffs = np.polyfit(x, y_vals, 1)
trend_line = np.polyval(coeffs, x)
direction = "decreasing 📉" if coeffs[0] < 0 else "increasing 📈"

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Coverage trend
axes[0].plot(year_df_plot['Year'], y_vals, marker='o', color='#2d6a4f',
             linewidth=2, markersize=7, label='Forest Coverage (%)')
axes[0].plot(year_df_plot['Year'], trend_line, '--', color='red',
             linewidth=1.5, label=f'Trend (slope={coeffs[0]:.3f}%/yr)')
axes[0].fill_between(year_df_plot['Year'], y_vals, alpha=0.15, color='#2d6a4f')
axes[0].set_title('Forest Coverage Trend Over Years', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Forest Coverage (%)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(alpha=0.3)

# Cumulative loss
axes[1].bar(year_df_plot['Year'], year_df_plot['Cumulative Loss (ha)'],
            color='#d62728', alpha=0.75, edgecolor='black', linewidth=0.5)
axes[1].set_title('Cumulative Forest Loss Since Baseline (ha)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Cumulative Loss (ha)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cumulative_loss_trend.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Summary stats table
total_loss_ha   = float(year_df_plot['Cumulative Loss (ha)'].iloc[-1])
total_loss_pct  = float(year_df_plot['Cumulative Loss (%)'].iloc[-1])
avg_annual_rate = defo_df['Deforestation Rate (%)'].mean() if len(defo_df) > 0 else 0
max_loss_period = defo_df.loc[defo_df['Deforestation Rate (%)'].idxmax(), 'Period'] if len(defo_df) > 0 else 'N/A'

summary_stats = pd.DataFrame({
    'Statistic': [
        'Baseline Forest Area (ha)',
        'Final Year Forest Area (ha)',
        'Total Forest Loss (ha)',
        'Total Forest Loss (%)',
        f'Coverage Trend',
        'Avg Annual Deforestation Rate (%)',
        'Worst Deforestation Period',
    ],
    'Value': [
        f"{baseline_ha:,.1f}",
        f"{year_df_plot.iloc[-1]['Est. Forest Area (ha)']:,.1f}",
        f"{total_loss_ha:,.1f}",
        f"{total_loss_pct:.2f}%",
        direction,
        f"{avg_annual_rate:.2f}%",
        max_loss_period,
    ]
})

print("\n========== HASDEO FOREST DEFORESTATION SUMMARY ==========")
display(summary_stats)
summary_stats.to_csv(os.path.join(OUTPUT_DIR, 'deforestation_summary.csv'), index=False)
print("\nSaved: deforestation_summary.csv, cumulative_loss_trend.png")
